In [2]:

import os
import re 

from pathlib import Path

# from dotenv import load_dotenv

import google.auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload

import duckdb
import pandas as pd
import pygsheets

import math

from sqlalchemy import create_engine

import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('ggplot')

ModuleNotFoundError: No module named 'duckdb'

#### ToDo:
-    Agregar bootstraping a calculo de mediana

In [ ]:
#load_dotenv('.env')
#%load_ext sql

In [ ]:
def max_rows(num):
    pd.set_option('display.max_rows',num)
    return f"Numero maximo de filas: {num}"

In [ ]:
def max_cols(num):
    pd.set_option('display.max_columns',num)
    return f"Numero maximo de columnas: {num}"

In [ ]:
max_rows(150)
max_cols(25)

'Numero maximo de columnas: 25'

In [ ]:
# carpeta donde está el notebook
base_dir = Path.cwd()
creds_file = base_dir / "cortes_gobernador" / "certs" / "GCPservice.json"

gc = pygsheets.authorize(service_file=str(creds_file))

In [ ]:
# Abre la hoja usando el ID es más seguro y rápido
sh = gc.open_by_key('1cmBjNaMReIYTQrIrehwlZ4nCJR7eYlVvOu_5-qbheXg')

# Selecciona la primera pestaña
wks = sh.sheet1

# Descarga los datos a tu DataFrame de Pandas
df_pytsh = wks.get_as_df(has_header=True)

# Muestra las primeras filas para confirmar que funcionó
df_pytsh.head()


,Centro,Serie,Placa,Linea,Fecha,TiempoEspera,InicioS1,FinS1,TecnicoS1,InicioS2,FinS2,TecnicoS2,...,Marca,SubMarca,Servicio,Cilindros,Carroceria,MetodoPrueba,TEMP_MOT,MIN_RPM_DIESEL,MAX_RPM_DIESEL,OPACIDAD,PBV,TipoPrueba
0,0,VF37R9HE1HJ582971,JV68939,2,2024-01-29 18:10:00,15,2024-01-29 18:19:00,2024-01-29 18:20:00,Jonhatan Escoto Enriquez,2024-01-29 18:20:00,2024-01-29 18:26:00,Jonhatan Escoto Enriquez,...,PEUGEOT,PARTNER HDI,SERVICIO DE CARGA,4 cil,VEHICULO UTILITARIO DEPORTIVO (SUV),Opacidad,96,748,4852,1.06,1,
1,0,MMBNG45K0FDZ00122,JT78273,2,2024-01-29 14:29:00,18,2024-01-29 14:39:00,2024-01-29 14:39:00,Jonhatan Escoto Enriquez,2024-01-29 14:39:00,2024-01-29 14:47:00,Erick Hernandez Rodriguez,...,MITSUBISHI,L200 TDI,SERVICIO DE CARGA,4 cil,PICKUP,Opacidad,92,701,5028,0.86,1,
2,0,VF3VFAHX1JZ001396,JX05500,2,2024-01-29 9:33:00,10,2024-01-29 9:36:00,2024-01-29 9:36:00,Jose de Jesus Guzman Nuñez,2024-01-29 9:36:00,2024-01-29 9:43:00,Jose de Jesus Guzman Nuñez,...,PEUGEOT,EXPERT VU,SERVICIO DE CARGA,4 cil,PANEL / VAN,Opacidad,287,747,3685,0.01,1,
3,0,VF37R9HE2HJ520592,JX17670,2,2024-01-25 10:11:00,16,2024-01-25 10:13:00,2024-01-25 10:15:00,Isaac Martinez Gonzalez,2024-01-25 10:15:00,2024-01-25 10:28:00,Isaac Martinez Gonzalez,...,PEUGEOT,PARTNER HDI,SERVICIO DE CARGA,4 cil,VEHICULO UTILITARIO DEPORTIVO (SUV),Opacidad,93,751,4784,0.51,1,
4,0,WV1CDASE3KX001661,JW42261,2,2024-01-25 9:48:00,14,2024-01-25 9:54:00,2024-01-25 9:54:00,Angel Ernesto Vargas Aguirre,2024-01-25 9:54:00,2024-01-25 10:01:00,Angel Ernesto Vargas Aguirre,...,VW,CADDY 2.0L TDI,SERVICIO DE CARGA,4 cil,VEHICULO UTILITARIO DEPORTIVO (SUV),Opacidad,91,830,2588,0.72,1,


In [ ]:
df_pytsh.columns

Index(['Centro', 'Serie', 'Placa', 'Linea', 'Fecha', 'TiempoEspera',
       'InicioS1', 'FinS1', 'TecnicoS1', 'InicioS2', 'FinS2', 'TecnicoS2',
       'TecnicoEgreso', 'Resultado', 'MotivoFracaso', 'NumeroTransaccion',
       'Tipo', 'TarjetaCirculacion', 'Odometro', 'Modelo', 'Clase',
       'Combustible', 'Marca', 'SubMarca', 'Servicio', 'Cilindros',
       'Carroceria', 'MetodoPrueba', 'TEMP_MOT', 'MIN_RPM_DIESEL',
       'MAX_RPM_DIESEL', 'OPACIDAD', 'PBV', 'TipoPrueba'],
      dtype='str')

In [ ]:
connection_string_wep = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=LiamThinkPad;DATABASE=wep_tables_feb_2023;trusted_connection=yes'
con_wep = create_engine('mssql+pyodbc:///?odbc_connect={}'.format(connection_string_wep))

ModuleNotFoundError: No module named 'pyodbc'

In [ ]:
#Considerar eliminar
connection_string_pop = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=LiamThinkPad;DATABASE=cortes_gobernador;trusted_connection=yes'
con_pop = create_engine('mssql+pyodbc:///?odbc_connect={}'.format(connection_string_pop))

In [ ]:
df_wep = pd.read_sql_table('diesel_specs_feb_2023',
                            con = con_wep)

In [ ]:
df_pytsh.info()

In [ ]:
df_pytsh[['Combustible','Estilodelvehículo','Transmisión','Servicio','Cilindros']] = df_pytsh[['Combustible','Estilodelvehículo','Transmisión','Servicio','Cilindros']].astype('category')
df_pytsh[['Marca','Submarca','DIESELMARCA','SUBMARCADIÉSEL']] = df_pytsh[['Marca','Submarca','DIESELMARCA','SUBMARCADIÉSEL']].astype('string') 

In [ ]:
df_wep.info()

In [ ]:
df_wep[['Marca','Submarca','submarcaDiesel']] = df_wep[['Marca','Submarca','submarcaDiesel']].astype('string')

## Para definir pruebas diesel:
- Realizar cruce con Submarcas diesel presentes en tablas proporcionadas por WEP
- Filtrar con tipo de combustible = `DIESEL`


In [ ]:
# Interseccion de lista de submarcas diesel verificadas con lista de submarcas diesel proporcionadas por wep

list_submarcas_vision = df_pytsh['Submarca'].unique()
list_submarcas_twep = df_wep['Submarca'].unique()

list_submarcas_inter = [value for value in list_submarcas_vision if value in list_submarcas_twep]

In [ ]:
len(list_submarcas_inter)

**Se podria asumir que aquellas submarcas que no se encuentran en la interseccion de las pruebas vehiculares con las submarcas de tablas de WEP, son aquellas que presentan confusion al momento de elegir el registro para el tipo de combustible o que ya no existen en las tablas de WEP**

In [ ]:
# Elementos del conjunto de pruebas diesel que no se encuentran en la interseccion
df_pytsh['Submarca'].where(~df_pytsh['Submarca'].isin(list_submarcas_inter)).dropna(how = 'all').value_counts().head(50)

In [ ]:
df_pytsh_inter = df_pytsh.where(df_pytsh['Submarca'].isin(list_submarcas_inter) & (~df_pytsh['MAXRPMDIESEL'].isin([0,3999,5999]))).dropna(how = 'all')
df_pytsh_inter.shape

In [ ]:
pop_table = pd.DataFrame(df_pytsh_inter[['Marca','Submarca','Modelo']].groupby(['Marca','Submarca','Modelo']).value_counts(ascending = False)).reset_index()

In [ ]:
pop_table.columns

In [ ]:
pop_table.info()

In [ ]:
pop_table.rename(columns = {0:'Population'}, inplace = True)

In [ ]:
pop_data_hist = sns.histplot(data = pop_table['Population'].loc[pop_table['Population'] > 0],
                             binwidth = 20,
                             stat = 'percent',
                             kde = True,
                             )

pop_data_hist.set_xlim(0,400)
pop_data_hist.set_xticks(range(0,382,25))
for i in pop_data_hist.containers:
        pop_data_hist.bar_label(i, fontsize = 7, fmt = '%.2f')

In [ ]:
pop_table['Population'].loc[pop_table['Population'] > 0].describe()

In [ ]:
Q3 = pop_table['Population'].loc[pop_table['Population'] > 0].quantile(0.75)

In [ ]:
# Cuartil Q3 
pop_table['Population'].loc[pop_table['Population'] >= Q3]

In [ ]:
pop_data_hist = sns.histplot(data = pop_table['Population'].loc[pop_table['Population'] >= Q3],
                             binwidth = 20,
                             stat = 'percent',
                             kde = True,
                             )

pop_data_hist.set_xlim(0,400)
pop_data_hist.set_xticks(range(0,401,25))
for i in pop_data_hist.containers:
        pop_data_hist.bar_label(i, fontsize = 7, fmt = '%.2f')

In [ ]:
pop_table.loc[pop_table['Population'] >= Q3].describe()

In [ ]:
# Dentro de conjunto de datos del Q3, analizar aquellos del Cuartil 3, sera denominado q3
q3 = pop_table['Population'].loc[pop_table['Population'] >= Q3].quantile(0.75)

In [ ]:
pop_data_hist = sns.histplot(data = pop_table['Population'].loc[pop_table['Population'] >= q3],
                             binwidth = 20,
                             stat = 'percent',
                             kde = True,
                             )

pop_data_hist.set_xlim(0,400)
pop_data_hist.set_xticks(range(0,381,25))
for i in pop_data_hist.containers:
        pop_data_hist.bar_label(i, fontsize = 7, fmt = '%.2f')

In [ ]:
pop_table['Population'].loc[pop_table['Population'] >= q3].describe()

In [ ]:
pop_table['Population'].loc[pop_table['Population'] > q3].groupby(pd.cut(pop_table['Population'],[0,98,200,300,400])).count()

In [ ]:
# Z-score: 80%:1.28, 85%:1.44, 90%:1.65, 95%:1.96, 99%:2.58 

Z = 2.576  # Z-score para 95% confidence level
σ = 5  # desviación estándar estimada
E = 1  # margen de error deseado
N = 1  # tamaño de población

list_pop = range(0,401,10)
# Calculate the required sample size
n = math.ceil((Z**2 * σ**2 * N) / ((Z**2 * σ**2) + (E**2 * (N-1))))
final = []
for pop in list_pop:
    result = math.ceil((Z**2 * σ**2 * pop) / ((Z**2 * σ**2) + (E**2 * (pop-1))))
    final.append(result)

print(list(zip(list_pop,final)))

sample_df = pd.DataFrame({'Pop':list_pop,'Sample':final})
#print(f"The required sample size is {n}")

In [ ]:
sns.scatterplot(y = sample_df['Sample'], x=sample_df['Pop'],hue=sample_df['Pop'])

### ~20% de las submarcas utilizadas durante pruebas (aquellas que tienen interseccion con las tablas de wep) tienen un tamanio de muestra de alrededor de 100 inspecciones, por fines practicos se trabajara con el 20% de los datos del Q3

- En pop_table con `Population` >= `90` asignar `Z_Score` = `2.576`
- En pop_table con `Population` >= `80` & <= `89` asignar `Z_Score` = `1.96`
- En pop_table con `Population` >= `54` & < `80` asignar `Z_Score` = `1.65`
- En pop_table con `Population` >= `27` & < `54` asignar `Z_Score` = `1.44`
- En pop_table de manera general asignar `E_Margin` = `1`
- En pop_table de manera general asignar `Std_Dev` = `5`
- En pop_table `Req_Population` se calcula de manera automatica

### Todo esto se hace para calcular el tamanio de muestra necesario para tomarlo como parametro de muestra aleatoria en cada submarca, ese tamanio de muestra se almacena en el atributo `Req_Population`, posteriormente habra otro atributo llamado `RPM_Median` en donde se almacenaran las medianas de RPM para cada submarca

In [ ]:
pop_table['Z_Score'] = 0
pop_table['Std_Dev'] = 5
pop_table['E_Margin'] = 1

pop_table[['Z_Score','Std_Dev','E_Margin']] = pop_table[['Z_Score','Std_Dev','E_Margin']].astype('float64')

In [ ]:
list_reg_99 = pop_table.loc[pop_table['Population'] >= 90].index
list_reg_95 = pop_table.loc[pop_table['Population'].between(80,89,inclusive='both')].index
list_reg_90 = pop_table.loc[pop_table['Population'].between(54,80,inclusive='both')].index
list_reg_85 = pop_table.loc[pop_table['Population'].between(27,53,inclusive='both')].index
current_working_regs = [list_reg_99,list_reg_95,list_reg_90,list_reg_85]
flattened_list = [item for sublist in current_working_regs for item in sublist]

In [ ]:
pop_table.loc[list_reg_99,'Z_Score'] = 2.576
pop_table.loc[list_reg_95,'Z_Score'] = 1.96
pop_table.loc[list_reg_90,'Z_Score'] = 1.65
pop_table.loc[list_reg_85,'Z_Score'] = 1.44

In [ ]:
# Req_Population se utilza como parametro para el tamanio de muestra aleatoria

pop_table['Req_Population'] = pop_table['Z_Score']**2 * pop_table['Std_Dev']**2 * pop_table['Population']\
                             / ((pop_table['Z_Score']**2 * pop_table['Std_Dev']**2)\
                             + (pop_table['E_Margin']**2 * (pop_table['Population']-1)))

In [ ]:
pop_table.loc[pop_table['Population'] >= 27]

In [ ]:
pop_table['Req_Population'] = pop_table['Req_Population'].fillna(0)

In [ ]:
pop_table['Req_Population'] = pop_table['Req_Population'].apply(lambda x: math.ceil(x)) 

In [ ]:
pop_table.rank(method='dense',numeric_only=True).sort_values(by = 'Population', ascending = False)

In [ ]:
pop_table['Ranking'] = pop_table['Population'].rank(method='dense',ascending=False)

In [ ]:
data = pop_table.sort_values(by='Population',ascending=False)

In [ ]:
with con_pop.connect():
    data.to_sql(
        'pop_table',
        con=con_pop,
        if_exists='replace',
        index=True
    )

#### ToDo:
- Hacer grafica que muestre la media de RPMs por cada submarca y pueda actualizarse a traves del tiempo
    - Debe tomar el tamanio de muestra para hacer la seleccion aleatoria
    - Desplegar la media de RPMs para esa submarca con el modelo correspondiente

In [ ]:
data['Z_Score'].value_counts()

In [ ]:
data_work_indexes = data[['Marca','Submarca','Modelo','Population','Req_Population']].where((data['Z_Score'] != 0)).dropna(how = 'all').index.tolist()

In [ ]:
data_2 = data[['Marca','Submarca','Modelo','Population','Req_Population','Z_Score','Ranking']].loc[data_work_indexes,]

In [ ]:
rpm_list = []
for x in data_2.index:
    median = df_pytsh_inter.where((df_pytsh_inter['Submarca'] == data_2.loc[x,'Submarca']) & (df_pytsh_inter['Modelo'] == data_2.loc[x,'Modelo'])).dropna(how = 'all').sample(data_2.loc[x,'Req_Population'])['MAXRPMDIESEL'].median()
    mean = df_pytsh_inter.where((df_pytsh_inter['Submarca'] == data_2.loc[x,'Submarca']) & (df_pytsh_inter['Modelo'] == data_2.loc[x,'Modelo'])).dropna(how = 'all').sample(data_2.loc[x,'Req_Population'])['MAXRPMDIESEL'].mean()
    #var = df_pytsh_inter.where((df_pytsh_inter['Submarca'] == data_2.loc[x,'Submarca']) & (df_pytsh_inter['Modelo'] == data_2.loc[x,'Modelo'])).dropna(how = 'all').sample(data_2.loc[x,'Req_Population'])['MAXRPMDIESEL'].var().round(2)
    std = df_pytsh_inter.where((df_pytsh_inter['Submarca'] == data_2.loc[x,'Submarca']) & (df_pytsh_inter['Modelo'] == data_2.loc[x,'Modelo'])).dropna(how = 'all').sample(data_2.loc[x,'Req_Population'])['MAXRPMDIESEL'].std(ddof = 1)
    rpm_list.append((x,median,math.ceil(mean),round(std)))
 

In [ ]:
data_rpm = pd.DataFrame(rpm_list,columns=['index','rpm_median','rpm_mean','std'])


In [ ]:
data_rpm = data_rpm.set_index('index')

In [ ]:
data_2

In [ ]:
data_2 = data_2.join(data_rpm)

In [ ]:
data_2 = data_2.astype({'Modelo':'int','Ranking':'int'})

In [ ]:
data_2

In [ ]:
#data_2.to_csv('diesel_rpm_stats.csv',encoding='latin-1')

In [ ]:
data_2.where(
    (data_2['Submarca'] == 'L200 TDI')\
    & (data_2['Modelo'] == 2015))\
    .dropna(how = 'all')

### Actualizar gsheet de pop_table

In [ ]:
creds_file_pipeline = r'C:\PythonProjects\GCPserviceAccounts\drev-374720-4417086df6ad.json'

In [ ]:
gc_ppl = pygsheets.authorize(service_file=creds_file_pipeline)

In [ ]:
sh_pop = gc_ppl.open('pop_table')
wks_pop = sh_pop.sheet1

In [ ]:
wks_pop.clear(fields="*")

In [ ]:
wks_pop.set_dataframe(
    df=pop_table,
    start=(1,1),
    copy_index=False,
    copy_head=True,
    fit=True
)

### Actualizar gsheet de diesel_stats

In [ ]:
sh_stats = gc_ppl.open('diesel_stats')
wks_stats = sh_stats.sheet1

In [ ]:
wks_stats.clear(fields="*")

In [ ]:
wks_stats.set_dataframe(
    df=data_2,
    start=(1,1),
    copy_index=False,
    copy_head=True,
    fit=True
)

#### Actualizar gsheet de seleccion_motores

In [ ]:
sh_mot = gc_ppl.open('seleccion_motores')
wks_mot = sh_mot.sheet1

In [ ]:
seleccion_motores = pd.read_csv(r"C:\Users\Liam_\Downloads\seleccion_motores.csv")

In [ ]:
wks_mot.set_dataframe(
    df=seleccion_motores,
    start=(1,1),
    copy_index=False,
    copy_head=True,
    fit=True
)

### Crear imagenes de distribuciones 

In [ ]:
graph = sns.histplot(
    df_pytsh_inter.where(
        (df_pytsh_inter['Submarca'] == 'L200 TDI')\
            & (df_pytsh_inter['Modelo'] == 2020)
            )['MAXRPMDIESEL'],
            kde=True,
            bins=20,
            stat='count',
            element='bars',
            fill=True,
            color='#133250',
            cbar=False
            #kde_kws={'bw_adjust':0.25},
            )
graph.set_title("Distribución de inspecciones de\n<submarca> <modelo>\nen banda de RPMs")
graph.set_ylabel("Cantidad de inspecciones")
graph.set_xlabel("Revoluciones por minuto")

sns.set(rc = {"figure.figsize":(8,8)})

for i in graph.containers:  
    graph.bar_label(
        container = i,
        label_type = "edge")

#fig = graph.get_figure() 
#fig.savefig("graph.png")

In [ ]:
for vehicle in data_2.index:
    graph = sns.histplot(
        df_pytsh_inter.where(
            (df_pytsh_inter['Submarca'] == data_2.loc[vehicle,'Submarca'])\
                & (df_pytsh_inter['Modelo'] == data_2.loc[vehicle,'Modelo'])
                )['MAXRPMDIESEL'],
                kde=True,
                bins=20,
                stat='count',
                element='bars',
                fill=True,
                color='#133250'
                #kde_kws={'bw_adjust':0.25},
                )
    graph.set_title(f"Distribución de inspecciones de\n{data_2.loc[vehicle,'Submarca']} {data_2.loc[vehicle,'Modelo']}\nen banda de RPMs")
    graph.set_ylabel("Cantidad de inspecciones")
    graph.set_xlabel("Revoluciones por minuto")

    sns.set(rc = {"figure.figsize":(8,8)})

    for i in graph.containers:  
        graph.bar_label(
            container = i,
            label_type = "edge")
    fig = graph.get_figure() 
    fig.savefig(f"Imagenes/{data_2.loc[vehicle,'Submarca'].replace('/','_')},{data_2.loc[vehicle,'Modelo']}.png")
    plt.clf()

### Subir/Actualizar imagenes de distribuciones a carpeta de google drive

In [ ]:
def get_file_id(service, folder_id, filename):
    response = service.files().list(q=f"'{folder_id}' in parents and name='{filename}'").execute()
    files = response.get('files', [])
    
    if files:
        return files[0]['id']
    else:
        return None
    

def upload_folder_to_drive(folder_path, target_folder_id):
    creds, _ = google.auth.load_credentials_from_file(
        filename="C:\\PythonProjects\\GCPserviceAccounts\\drev-374720-4417086df6ad.json" # Anadir a variable local
    )
    
    try:
        # Create Google Drive API client
        service = build("drive", "v3", credentials=creds)

        # Iterate through files in the local folder
        for filename in os.listdir(folder_path):

            file_path = os.path.join(folder_path, filename)

            # Get the file ID if it exists in the target folder
            file_id = get_file_id(service, target_folder_id, filename)

            # Define file metadata with parent folder ID
            file_metadata = {
                "name": filename,
                "mimeType": "PNG",  
                "parents": [target_folder_id] # parents cuando no hay informacion inicial
            }

            # Create media file upload
            media = MediaFileUpload(file_path, resumable=True)

            if file_id:

                file_metadata = {
                "name": filename,
                "mimeType": "PNG",  
                "addParents": [target_folder_id]
                }
                # If the file exists, update it
                file = (
                    service.files()
                    .update(fileId=file_id, body=file_metadata, media_body=media, fields="id")
                    .execute()
                )
                print(f'File with ID: "{file.get("id")}" has been updated in folder with ID: "{target_folder_id}".')

            else:

                file_metadata = {
                "name": filename,
                "mimeType": "PNG",  
                "parents": [target_folder_id]
                }
                # If the file doesn't exist, create it
                file = (
                    service.files()
                    .create(body=file_metadata, media_body=media, fields="id")
                    .execute()
                )
                print(f'File with ID: "{file.get("id")}" has been uploaded to folder with ID: "{target_folder_id}".')
            
    except HttpError as error:
        print(f"An error occurred: {error}")

folder_path = "Imagenes"
target_folder_id = "1VTUfbDEhbBbHtOYSAZfTeVe3mvqx8XQo" # Variable de entorno
image_metadata = upload_folder_to_drive(folder_path, target_folder_id)

### Conseguir metadatos de imagenes de graficas y actulizar gsheet de distribuciones_rpms

In [ ]:
def get_images_metadata(folder_id: str) -> list:
    creds, _ = google.auth.load_credentials_from_file(
        filename="C:\\PythonProjects\\GCPserviceAccounts\\drev-374720-4417086df6ad.json"
    )
    service = build('drive', 'v3', credentials=creds)
    
    response = service.files().list(q=f"'{folder_id}' in parents and mimeType='image/png'",
                                    pageSize=100).execute()

    files = response.get('files', [])
    next_page_token = response.get('nextPageToken')

    while next_page_token:
        response = service.files().list(q=f"'{folder_id}' in parents and mimeType='image/png'",
                                        pageSize=100,
                                        pageToken=next_page_token).execute()
        files.extend(response.get('files', []))
        next_page_token = response.get('nextPageToken')

    return files

image_metadata = get_images_metadata('1VTUfbDEhbBbHtOYSAZfTeVe3mvqx8XQo') # variable de entorno

In [ ]:
list_image_metadata = []
for pos in range(0,len(image_metadata)):
    list_image_metadata.append((image_metadata[pos].get('name'),image_metadata[pos].get('id')))

image_metadata_df = pd.DataFrame(list_image_metadata, columns=['name','id'])


In [ ]:
# Transformar image_metadata_df

image_metadata_df[['Submarca','Modelo']] = image_metadata_df['name'].str.extract(pat=r'([^,]+),([^\/.]+)',expand=True)
image_metadata_df = image_metadata_df[['id','name','Submarca','Modelo']]
image_metadata_df['Submarca'].replace(regex='_',value='/',inplace=True)
image_metadata_df['name'].replace(regex='_',value='/',inplace=True)

In [ ]:
# Actualizar gsheet de distribuciones_rpms
creds_file_img = r"C:\PythonProjects\GCPserviceAccounts\drev-374720-ae98225549f4.json"
sh_img = gc_ppl.open('distribuciones_rpms')
wks_img = sh_img.sheet1
wks_img.clear(fields='*')
wks_img.set_dataframe(
    df = image_metadata_df,
    start=(1,1),
    copy_index=False,
    copy_head=True,
    fit=True
)